# Homework #1

In [1]:
from dotenv import load_dotenv
load_dotenv()

from minsearch import Index
from openai import OpenAI

import rag_helper

In [2]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [ ]:
## Q1. How many lesson pages 
print(f"Number of lesson Pages: {len(files)}")

Number of lession Pages: 72


## Question 1
Answer: Number of lession Pages: 72


In [5]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

index = build_index(documents)

In [6]:
question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(
    question,
    #boost_dict={"question": 2.0, "section": 0.5},
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# Q2: 01-agentic-rag/lessons/14-agentic-loop.md 
[print(doc["filename"]) for doc in search_results]

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


[None, None, None, None, None]

## Question 2 
Answer: 01-agentic-rag/lessons/14-agentic-loop.md 

## Rag


In [7]:
openai_client = OpenAI()
assistant = rag_helper.RAGBase(
    index=index,
    llm_client=openai_client,
)

In [8]:
answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Input Tokens used: {input_tokens}")
print(answer)
# Q3 - 7032 input tokens

Input Tokens used: 7036
It keeps calling the model inside a `while True` loop.

Each turn:

1. Send the full `messages` history to the model.
2. Check the response for any `function_call` items.
3. If there are function calls, run the tool and append the tool output to `messages`.
4. If there are no function calls, `break` out of the loop.

So the stopping condition is: **no function calls in the latest response**.


## Question 3 
Answer: 7032 input tokens

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print("Chunks: ", len(chunks))
# Q4 chunks 295

Chunks:  295


## Question 4
Answer: 295 chunks

In [10]:
index_chunked = build_index(chunks)

question = "How does the agentic loop keep calling the model until it stops?"
search_results = index_chunked.search(
    question,
    #boost_dict={"question": 2.0, "section": 0.5},
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [11]:
openai_client = OpenAI()
assistant = rag_helper.RAGBase(
    index=index_chunked,
    llm_client=openai_client,
)

answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Chunked Input Tokens used: {input_tokens}")
print(answer)
# chunked input tokens 2221 / 3x fewer

Chunked Input Tokens used: 2221
The loop keeps calling the model with a `while True` loop and stops when the model returns a turn with **no `function_call` items**.

In the code, that is tracked with:

```python
has_function_calls = False
...
if item.type == "function_call":
    ...
    has_function_calls = True
...
if has_function_calls == False:
    break
```

So the agentic loop:
1. sends the current messages to the model,
2. runs any tool calls the model asks for,
3. adds the tool results back into `messages`,
4. calls the model again,
5. and exits only when a response comes back without any function calls.




## Question 5
Answer: 2221 input tokens.  ~ 3x fewer

In [12]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [13]:

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index_chunked.search(
        query,
        num_results=5,
        #boost_dict={'question': 3.0, 'section': 0.5},
        #filter_dict={'course': 'llm-zoomcamp'}
    )

In [14]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

In [17]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [18]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received
